In [0]:
from pyspark.sql.functions import current_timestamp

# 1. Setup paths in your Volume
volume_base = "/Volumes/practice/raw/kaggle_files"
checkpoint_path = "/Volumes/practice/checkpoints/netflix_cp"
schema_path = "/Volumes/practice/checkpoints/netflix_schema"

# 2. Read with Auto Loader
df = (spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format", "csv")
      .option("cloudFiles.schemaLocation", schema_path) 
      .load(f"{volume_base}/")
      .withColumn("_ingested_at", current_timestamp())
     )

# 3. Write with Checkpoint (This fixes your error)
(df.writeStream
  .format("delta")
  .option("checkpointLocation", checkpoint_path) # <--- ADD THIS
  .outputMode("append")
  .option("mergeSchema", "true")
  .trigger(availableNow=True)
  .toTable("practice.bronze.netflix_titles"))


